  ### Retrieving Data from the API

We first import Python's requests library, which allows us to make HTTP requests. We then define the API endpoint we'll be interacting with, setting API_ENDPOINT to https://api.water.noaa.gov/hefs/. Here's how we do it:

In [ ]:
# start by importing the requests library
import requests
# import the pprint library to make the output more readable
from pprint import pprint
# import pandas for easier quantile calculations and formatting
import pandas as pd
# import scipy for quantile calculations
from scipy.stats.mstats import mquantiles

# define the api-endpoint
API_ENDPOINT = "https://api.water.noaa.gov/hefs"
# define location id
LOCATION_ID = "GOSO3"
# define parameter id
PARAMETER_ID = "QINE"

Now, we will grab the hydrograph quantiles for the specified location id and parameter id. To do this we will use the hydrograph quantiles endpoint as follows:

`/v1/hydrograph-quantiles/?location_id={LOCATION_ID}&parameter_id={PARAMETER_ID}`

In [ ]:
# create a series request with location_id, parameter_id, start_date_date, and limit filters
uri = API_ENDPOINT + f"/v1/hydrograph-quantiles/?location_id={LOCATION_ID}&parameter_id={PARAMETER_ID}"
# get the response
response = requests.request("GET", uri)
# print the response
pprint(response.json())

Finally, we plot the different timestep quantiles by value for different exceedance probabilities (inverse of the p value). This is the probability of a value exceeding a specific value at a given time.

In [ ]:
import matplotlib.pyplot as plt
import datetime

# dictionary to different values per timestep
step_vals = {}
# array of timesteps
timesteps = []

responseJ = response.json()
# iterate through length of mquantile response
for result in responseJ['value_set']:
    # format datetime
    datetime_str = f"{result['valid_datetime']}"
    datetime_str = datetime_str.replace('T', " ")
    datetime_str = datetime_str.replace('Z', "")
    datetime_str = datetime.datetime.strptime(datetime_str, "%Y-%m-%d %H:%M:%S")
    # check if step vals has a dictionary key that is datetime
    if(not step_vals.get(datetime_str)):
      timesteps.append(datetime_str)
      step_vals[datetime_str] = []
    # add value to stepvals array for corresponding datetime
    step_vals[datetime_str] = result['quantile_values']
keys = list(step_vals.keys())
values = list(step_vals.values())
for i in range(len(values[0])):
  vallist = []
  for j in range(len(keys)):
    vallist.append(values[j][i])
  plt.plot(keys,vallist)

exceed = responseJ['metadata']['exceedance_quantiles']

# prepare for putting the legend outside of graph
ax = plt.subplot(111)
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])
plt.legend(exceed, title="Exceedance Probability",loc="center left", fontsize="small",bbox_to_anchor=(1, 0.5))
# set x graph label
plt.xlabel('Time')
# set y graph label
plt.ylabel('Value')
# set graph title
plt.title('Quantiles for Different Probabilities')
# rotate x text vertically for better visibility
plt.xticks(rotation='vertical')
# display the actaul plot
plt.show()

## Summary

In this notebook, we learned how to use the HEFS API to retrieve the latest ensemble forecast for a given location. We also learned how to calculate quantiles for the respective forecasts.

These techniques can be used to retrieve, filter, and paginate data for all HEFS ensemble forecasts.